# Exercises XP: Evaluating LLMs for Summarization

## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.

In [ ]:
# Part I. Setup (run once per runtime)
!pip -q install rouge_score==0.1.2 evaluate datasets "transformers<5" accelerate nltk --quiet

import os
os.environ["USE_TF"] = "0"                 # transformers: PyTorch only
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
import time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, transformers, evaluate
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
torch.manual_seed(0); random.seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| evaluate", evaluate.__version__, "| device:", DEVICE)

### Part II. Dataset loading and exploration
Dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail), config `3.0.0`. We map `article` → `prompt_text` and `highlights` → `prompt_title`.

Instead of downloading the full dataset (≈ 1.3 GB), we use the **Hugging Face datasets-server API**, as suggested in the instructions. It returns pages of up to 100 rows, so we fetch a few pages at random offsets of the `train` and `test` splits and save them as `train.csv` and `test.csv`. Those two files are then loaded with pandas and sampled (100 train / 50 test). If the API is unreachable, the notebook falls back to `datasets.load_dataset(..., streaming=True)`.

In [ ]:
import requests

API = "https://datasets-server.huggingface.co/rows"
DS = {"dataset": "abisee/cnn_dailymail", "config": "3.0.0"}

def fetch_split_to_csv(split, path, n_pages=3, page_size=100, seed=42):
    if os.path.exists(path):
        return
    rng = random.Random(seed)
    total = requests.get(API, params={**DS, "split": split, "offset": 0, "length": 1}, timeout=60).json()["num_rows_total"]
    rows = []
    for _ in range(n_pages):
        offset = rng.randrange(0, total - page_size)
        for attempt in range(4):
            try:
                r = requests.get(API, params={**DS, "split": split, "offset": offset, "length": page_size}, timeout=60)
                r.raise_for_status(); break
            except requests.RequestException:
                time.sleep(2 ** attempt)
        rows += [x["row"] for x in r.json()["rows"]]
    df = pd.DataFrame(rows).rename(columns={"article": "prompt_text", "highlights": "prompt_title"})
    df[["id", "prompt_text", "prompt_title"]].to_csv(path, index=False)
    print(f"{split}: {total} rows in the full split -> saved {len(df)} rows to {path}")

def fetch_with_datasets(split, path, n=300):
    from datasets import load_dataset
    ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split=split, streaming=True)
    rows = [x for _, x in zip(range(n), ds)]
    pd.DataFrame(rows).rename(columns={"article": "prompt_text", "highlights": "prompt_title"})[["id", "prompt_text", "prompt_title"]].to_csv(path, index=False)

for split, path in [("train", "train.csv"), ("test", "test.csv")]:
    try:
        fetch_split_to_csv(split, path)
    except Exception as exc:
        print(f"datasets-server failed ({exc}); streaming with datasets instead")
        fetch_with_datasets(split, path)

In [ ]:
train_path = 'train.csv'
test_path = 'test.csv'

def load_and_sample(path, n, seed=42):
    df = pd.read_csv(path)
    return df.sample(min(n, len(df)), random_state=seed).reset_index(drop=True)

train_df = load_and_sample(train_path, 100)
test_df = load_and_sample(test_path, 50)

first = train_df.iloc[0]
print("ARTICLE (prompt_text):\n", first["prompt_text"][:1500], "...\n")
print("REFERENCE SUMMARY (prompt_title):\n", first["prompt_title"])

In [ ]:
print("Train sample:", train_df.shape, "| Test sample:", test_df.shape)
display(train_df)
display(test_df)

In [ ]:
stats = pd.DataFrame({
    "article words": train_df["prompt_text"].str.split().str.len(),
    "summary words": train_df["prompt_title"].str.split().str.len(),
    "summary sentences": train_df["prompt_title"].str.count("\n") + 1,
}).describe().round(1)
stats

Articles average roughly 700 words, much longer than T5's 512-token input window and close to GPT-2's 1 024-token limit, so they will be **truncated**. Reference summaries (*highlights*) are 3–4 bullet sentences, roughly 50 words, written by journalists, separated by newlines and often **abstractive**: they rephrase rather than copy.

### Part III. Summarization with T5
- `batch_generator` yields mini-batches.
- `summarize_with_t5` uses `T5ForConditionalGeneration`, a GPU when one is available, the `"summarize: "` prefix, `generate()` and `decode(skip_special_tokens=True)`. It clears the CUDA cache and calls `gc.collect()` after each batch and at the end.

Generation settings are the same for every model: beam search with 4 beams, at most 64 new tokens (CNN/DM highlights are about 50 words), and `no_repeat_ngram_size=3` against loops.

In [ ]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 64):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device).eval()
    summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(["summarize: " + t for t in batch], max_length=512, truncation=True,
                           padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4,
                                 no_repeat_ngram_size=3, early_stopping=True)
        summaries += tokenizer.batch_decode(out, skip_special_tokens=True)
        del inputs, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return [s.strip() for s in summaries]

RUN_T5 = True
timings = {}
if RUN_T5:
    t0 = time.time()
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=4)
    timings["t5-small"] = time.time() - t0
    print(f"t5-small: {len(train_summaries_t5)} summaries in {timings['t5-small']:.0f}s")
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'].str.slice(0, 200) + "...",
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head(10))
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

### Part IV. Accuracy evaluation (toy, likely near zero)
Naive accuracy = the share of generated summaries that are an **exact string match** with the reference.

In [ ]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
    # even a 'perfect copy' of the reference sentences fails if only the line breaks differ
    refs = train_df['prompt_title'].tolist()
    print("Accuracy of the references against themselves with newlines replaced by spaces:",
          compute_accuracy([r.replace("\n", " ") for r in refs], refs))
    # token-level 'accuracy': position-by-position word match (also meaningless for free text)
    tok_acc = np.mean([np.mean([a == b for a, b in zip(p.lower().split(), r.lower().split())] or [0])
                       for p, r in zip(train_summaries_t5, refs)])
    print(f"Position-wise word accuracy: {tok_acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")

**Why accuracy is (almost) zero.** Summarization is open-ended: there are many valid summaries of an article, and the reference is only one of them. An exact match requires the same words, in the same order, with the same punctuation and line breaks. Even the references themselves score **0** once their newlines are replaced by spaces.

A position-wise word match is no better: inserting one word shifts everything after it. Accuracy gives no partial credit and therefore cannot rank models. We need an **overlap-based** metric: ROUGE.

### Part V. ROUGE metric implementation
**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) counts the n-grams shared by the candidate and the reference summary:
- **ROUGE-N**: overlap of n-grams. ROUGE-1 counts single words (content), ROUGE-2 counts bigrams (fluency and phrasing).
- **ROUGE-L**: the longest common subsequence (word order, without requiring contiguity).
- **ROUGE-Lsum**: ROUGE-L computed *per sentence* and then combined. It needs the sentences to be separated by **newlines**.

For each variant we get precision (overlap / candidate n-grams), recall (overlap / reference n-grams) and F1; `evaluate` reports the F1.

**Preprocessing.** Model outputs are single strings with no line breaks. We split them into sentences with NLTK's `sent_tokenize` and join the sentences with `"\n"`, so that `rougeLsum` sees the sentence boundaries. ⚠️ The template's `normalize_text` joined the sentences with `""`, which glues the last word of one sentence to the first word of the next (`"sat.The"`) and breaks `rougeLsum`; we fix it below.

In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(str(text).strip())
    return "\n".join(sents)          # one sentence per line (needed by rougeLsum)

def compute_rouge_score(preds: List[str], refs: List[str], use_stemmer: bool = True, use_aggregator: bool = True):
    preds = [normalize_text(p) for p in preds]
    refs = [normalize_text(r) for r in refs]
    scores = rouge.compute(predictions=preds, references=refs, use_stemmer=use_stemmer, use_aggregator=use_aggregator)
    return {k: (round(float(v), 4) if use_aggregator else [round(float(x), 4) for x in v]) for k, v in scores.items()}

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check:")
print("  aggregated:", compute_rouge_score(test_preds, test_refs))
print("  per pair  :", compute_rouge_score(test_preds, test_refs, use_aggregator=False))
print("\nnormalize_text example:", repr(normalize_text("The cat sat. The dog ran away! Did it?")))

### Part VI. Understanding ROUGE scores

In [ ]:
refs = train_df["prompt_title"].tolist()

# 1. Exact match
print("1) predictions == references :", compute_rouge_score(refs, refs))

# 2. Null predictions
print("2) empty predictions          :", compute_rouge_score([""] * len(refs), refs))

# 3. Stemming
pairs = [("The kids were running fast", "The kid runs fast"),
         ("Prices increased sharply", "The price increases sharply")]
for p, r in pairs:
    print(f"3) {p!r} vs {r!r}: no stemmer {compute_rouge_score([p], [r], use_stemmer=False)['rouge1']}"
          f"  | with stemmer {compute_rouge_score([p], [r], use_stemmer=True)['rouge1']}  (ROUGE-1 F1)")

In [ ]:
# 4. N-gram analysis: increasing overlap with a reference sentence
reference = "the quick brown fox jumps over the lazy dog"
candidates = {
    "identical": "the quick brown fox jumps over the lazy dog",
    "same words, shuffled": "dog lazy the over jumps fox brown quick the",
    "one word changed": "the quick brown fox leaps over the lazy dog",
    "half the sentence": "the quick brown fox",
    "synonyms": "a fast auburn fox hops above a sleepy hound",
    "unrelated": "stock markets fell sharply on monday morning",
}
ngram_df = pd.DataFrame({name: compute_rouge_score([c], [reference], use_stemmer=False) for name, c in candidates.items()}).T
ngram_df

In [ ]:
# 5. Symmetry: swapping predictions and references
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

a = "the quick brown fox"
b = "the quick brown fox jumps over the lazy dog"
s_ab, s_ba = scorer.score(target=b, prediction=a), scorer.score(target=a, prediction=b)
sym = pd.DataFrame({
    "pred=short, ref=long (P/R/F)": {k: tuple(round(x, 3) for x in v) for k, v in s_ab.items()},
    "pred=long, ref=short (P/R/F)": {k: tuple(round(x, 3) for x in v) for k, v in s_ba.items()},
})
display(sym)
print("evaluate (F1) a->b:", compute_rouge_score([a], [b]), "\nevaluate (F1) b->a:", compute_rouge_score([b], [a]))

# on real data: t5-small summaries vs references, and swapped
if 'train_summaries_t5' in locals():
    print("\nt5-small vs refs :", compute_rouge_score(train_summaries_t5, refs))
    print("refs vs t5-small :", compute_rouge_score(refs, train_summaries_t5))

ANALYSIS_ROUGE

### Part VII. Comparing small and large models
- `t5-small` (60 M parameters) and `t5-base` (220 M) are encoder-decoder models. During pre-training they saw CNN/DailyMail with the `summarize:` prefix, so they are trained summarizers.
- `gpt2` (124 M) is a decoder-only language model that was **never trained to summarize**. Following the GPT-2 paper, we prompt it with `article + "\nTL;DR:"`.

GPT-2 has a hard context limit of 1 024 tokens (prompt + generated tokens). We truncate the article from the left side of the budget (we keep its beginning) so that `article + "TL;DR:" + max_new_tokens ≤ 1024`, pad on the left for batched generation, and keep only the newly generated tokens.

We also add the classic **Lead-3 baseline** (the first 3 sentences of the article), which is famously hard to beat on CNN/DM.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 64):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token          # GPT-2 has no pad token
    tokenizer.padding_side = "left"                    # decoder-only: pad on the left for batched generation
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

    suffix = "\nTL;DR:"
    suffix_ids = tokenizer(suffix)["input_ids"]
    max_article = model.config.n_positions - max_new_tokens - len(suffix_ids)    # 1024 - 64 - 4 tokens
    summaries = []
    for batch in batch_generator(texts, batch_size):
        prompts = []
        for t in batch:
            ids = tokenizer(t)["input_ids"][:max_article]                          # keep the beginning of the article
            prompts.append(tokenizer.decode(ids) + suffix)
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, no_repeat_ngram_size=3,
                                 early_stopping=True, pad_token_id=tokenizer.eos_token_id)
        new_tokens = out[:, inputs["input_ids"].shape[1]:]                         # drop the prompt
        for s in tokenizer.batch_decode(new_tokens, skip_special_tokens=True):
            s = s.strip().split("\n\n")[0].strip()                                  # stop at the first blank line
            summaries.append(s)
        del inputs, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    del model
    gc.collect()
    return summaries

def lead_3(texts):
    return [" ".join(sent_tokenize(t)[:3]) for t in texts]

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    scores = compute_rouge_score(df[pred_col].fillna("").tolist(), df[ref_col].tolist(), use_aggregator=False)
    out = df.copy()
    for metric, values in scores.items():
        out[f"{pred_col}_{metric}"] = values
    return out

In [ ]:
RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    articles = train_df["prompt_text"].tolist()
    results_df = train_df.copy()
    results_df["t5-small"] = train_summaries_t5

    t0 = time.time(); results_df["t5-base"] = summarize_with_t5(articles, model_name="t5-base", batch_size=4); timings["t5-base"] = time.time() - t0
    t0 = time.time(); results_df["gpt2"] = summarize_with_gpt2(articles, model_name="gpt2", batch_size=2); timings["gpt2"] = time.time() - t0
    t0 = time.time(); results_df["lead-3"] = lead_3(articles); timings["lead-3"] = time.time() - t0
    print({k: f"{v:.0f}s" for k, v in timings.items()})

    MODELS = ["t5-small", "t5-base", "gpt2", "lead-3"]
    rouge_scores = {m: compute_rouge_score(results_df[m].tolist(), results_df["prompt_title"].tolist()) for m in MODELS}
    for m, s in rouge_scores.items():
        print(f"{m:9s} {s}")

In [ ]:
# Per-row ROUGE for each model
per_row = results_df[["prompt_title"] + MODELS].copy()
for m in MODELS:
    per_row = compute_rouge_per_row(per_row, pred_col=m)
for m in MODELS:
    print(f"--- {m}")
    display(per_row[["prompt_title", m, f"{m}_rouge1", f"{m}_rouge2", f"{m}_rougeL"]].head(5))

### Part VIII. Comparing all models

In [ ]:
import pandas as pd

def compare_models(rouge_dict):
    # {model_name: rouge_scores_dict} -> DataFrame (one row per model), sorted by ROUGE-L
    return pd.DataFrame(rouge_dict).T.sort_values("rougeL", ascending=False)

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    return df[["prompt_title"] + pred_cols].rename(columns={"prompt_title": "reference"})

comparison = compare_models(rouge_scores)
comparison["avg words"] = [results_df[m].str.split().str.len().mean().round(1) for m in comparison.index]
comparison["time / article (s)"] = [round(timings[m] / len(results_df), 3) for m in comparison.index]
display(comparison)
display(compare_models_summaries(results_df, MODELS).head(8))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
comparison[["rouge1", "rouge2", "rougeL", "rougeLsum"]].plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Average ROUGE F1 on 100 CNN/DailyMail train articles"); axes[0].set_ylim(0, 0.5); axes[0].grid(axis="y", alpha=.3)
per_row[[f"{m}_rougeL" for m in MODELS]].rename(columns=lambda c: c.replace("_rougeL", "")).plot.box(ax=axes[1])
axes[1].set_title("Per-article ROUGE-L distribution"); axes[1].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

#### Modified accuracy metrics for summarization
Exact-match accuracy is useless, but we can make "accuracy" meaningful by turning it into a *pass rate*: a summary counts as correct if it is "good enough". We use two variants:
- **ROUGE-L ≥ τ**: the share of summaries whose ROUGE-L F1 reaches a threshold τ;
- **keyword recall accuracy**: the share of summaries that contain at least half of the reference's content words (no stopwords, stemmed).

In [ ]:
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
_stem, _stop = PorterStemmer(), set(stopwords.words("english"))

def content_words(text):
    return {_stem.stem(w) for w in nltk.word_tokenize(str(text).lower()) if w.isalnum() and w not in _stop}

def threshold_accuracy(rouge_l_values, tau=0.3):
    return float(np.mean(np.array(rouge_l_values) >= tau))

def keyword_recall_accuracy(preds, refs, min_recall=0.5):
    hits = [len(content_words(p) & content_words(r)) / max(len(content_words(r)), 1) >= min_recall for p, r in zip(preds, refs)]
    return float(np.mean(hits))

custom = pd.DataFrame({
    m: {"exact-match acc": compute_accuracy(results_df[m].tolist(), results_df["prompt_title"].tolist()),
        "acc (ROUGE-L ≥ 0.2)": threshold_accuracy(per_row[f"{m}_rougeL"], 0.2),
        "acc (ROUGE-L ≥ 0.3)": threshold_accuracy(per_row[f"{m}_rougeL"], 0.3),
        "keyword-recall acc (≥50%)": keyword_recall_accuracy(results_df[m], results_df["prompt_title"])}
    for m in MODELS}).T
custom.round(3)

In [ ]:
# Generalisation check on the held-out test sample (50 articles)
test_articles = test_df["prompt_text"].tolist()
test_preds = {
    "t5-small": summarize_with_t5(test_articles, "t5-small"),
    "t5-base": summarize_with_t5(test_articles, "t5-base"),
    "gpt2": summarize_with_gpt2(test_articles, "gpt2"),
    "lead-3": lead_3(test_articles),
}
test_comparison = compare_models({m: compute_rouge_score(p, test_df["prompt_title"].tolist()) for m, p in test_preds.items()})
test_comparison

In [ ]:
# A closer look at a few GPT-2 outputs (debugging)
for i in range(3):
    print(f"--- article {i}: {results_df['prompt_text'][i][:150]}...")
    print("REF :", results_df["prompt_title"][i].replace("\n", " | "))
    print("GPT2:", results_df["gpt2"][i])
    print("T5-B:", results_df["t5-base"][i], "\n")

ANALYSIS_WRAP